# Bóc tách code (Not supposed to run)

In [ ]:
class BCEWithNaNLoss(BCELoss):
    """
    Binary Cross Entropy Loss that handles NaN values by ignoring them in the gradient computation.
    Used for the uplift component where counterfactuals are not observed (represented as NaN).
    """
    def __init__(self, uplift=False):
        self.uplift = uplift
        self.clip_value = 1e-6

    def base_score(self, y_true):
        # Calculate base score (initial prediction) ignoring NaNs
        means = cp.nanmean(y_true, axis=0)
        means = cp.where(cp.isnan(means), 0, means)
        means = cp.clip(means, self.clip_value, 1 - self.clip_value)
        return cp.log(means / (1 - means))

`base_score` dùng để tìm initial prediction cho GB dựa trên average:

`cp.nanmean(y_true, axis=0)`: Tính giá trị trung bình (tỷ lệ positve) của từng cột (Control & Treatment)

`cp.clip()`: Kẹp giá trị trung bình vào khoảng $[\mathbf{0.000001}, \mathbf{0.999999}]$
- Nếu không sử dụng `clip` thì sẽ gây lỗi:
    - `means = 0`:
        $\frac{means}{1-means} = 0 \rightarrow log(0) = -\infty$ 
    - `means = 1`:
        $\frac{means}{1-means} = \frac{1}{0} \rightarrow \text{lỗi chia cho 0} \rightarrow log(+\infty) = +\infty$ 
- Sử dụng `clip` sẽ ép `means` trong khoảng $[\mathbf{0.000001}, \mathbf{0.999999}]$ -> `logit` luôn hữu hạn

`cp.log(means / (1 - means))`: Log-odd/Logit - Hàm ngược của Sigmoid
- Sử dụng hàm này để sau khi sigmoid, kết quả trả về = means (isomorphism)